In [3]:

#Librairies import
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import requests
import numpy as np
import json
import boto3
import os
import io
import re
from datetime import datetime
import time
import psycopg2
from dotenv import load_dotenv
load_dotenv()

True

Goal:
- Scrape data from destinations
- Get weather data from each destination
- Get hotels' info about each destination
- Store all the information above in a data lake
- Extract, transform and load cleaned data from your datalake to a data warehouse


In [4]:
#villes de références:
cities = [
    "Le Mont-Saint-Michel", "Saint Malo", "Bayeux", "Le Havre", "Rouen",
    "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon", "Annecy", "Grenoble", "Lyon",
    "Gorges du Verdon", "Bormes les Mimosas", "Cassis", "Marseille",
    "Aix en Provence", "Avignon", "Uzes", "Nimes", "Aigues Mortes",
    "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege",
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle"
]

df = pd.DataFrame()
df['id']= [id for id in range(len(cities))]
df['city']= cities

## Coordonnées GPS

In [5]:
#definie URL_Base
URL_BASE = 'https://nominatim.openstreetmap.org/'

# Check API status
response = requests.get(URL_BASE + 'status', headers={'User-Agent': 'trip-kayak/1.0'}, params={'format': 'json'})
if response.json()['message'] != 'OK':
    print('Error:', response.text)
    exit()
 
lat_list = []
lon_list = []
 
for city in cities:
    params = {'q': f"{city}, france", 'format': 'json', 'limit': 1}
    gps = requests.get(URL_BASE + 'search', headers={'User-Agent': 'trip-kayak/1.0'}, params=params).json()
 
    if isinstance(gps, list) and len(gps) > 0:
        lat_list.append(float(gps[0]['lat']))
        lon_list.append(float(gps[0]['lon']))
        print(f"{city}: {gps[0]['lat']}, {gps[0]['lon']}")
    else:
        lat_list.append(None)
        lon_list.append(None)
        print(f"{city}: not found — response: {gps}")
 
    time.sleep(1)
 
df = pd.DataFrame({'city': cities, 'latitude': lat_list, 'longitude': lon_list})
print(df)

Le Mont-Saint-Michel: 48.6355232, -1.5102571
Saint Malo: 48.6495180, -2.0260409
Bayeux: 49.2764624, -0.7024738
Le Havre: 49.4938975, 0.1079732
Rouen: 49.4404591, 1.0939658
Paris: 48.8588897, 2.3200410
Amiens: 49.8941708, 2.2956951
Lille: 50.6365654, 3.0635282
Strasbourg: 48.5846140, 7.7507127
Chateau du Haut Koenigsbourg: 48.2493820, 7.3439412
Colmar: 48.0777517, 7.3579641
Eguisheim: 48.0447968, 7.3079618
Besancon: 47.2380222, 6.0243622
Dijon: 47.3215806, 5.0414701
Annecy: 45.8992348, 6.1288847
Grenoble: 45.1875602, 5.7357819
Lyon: 45.7578137, 4.8320114
Gorges du Verdon: 43.7496562, 6.3285616
Bormes les Mimosas: 43.1506968, 6.3419285
Cassis: 43.2140359, 5.5396318
Marseille: 43.2961743, 5.3699525
Aix en Provence: 43.5298424, 5.4474738
Avignon: 43.9492493, 4.8059012
Uzes: 44.0121279, 4.4196718
Nimes: 43.8374249, 4.3600687
Aigues Mortes: 43.5661521, 4.1915400
Saintes Maries de la mer: 43.4515922, 4.4277202
Collioure: 42.5250500, 3.0831554
Carcassonne: 43.2130358, 2.3491069
Ariege: 42.9455

## Méteo

In [ ]:
OPEN_WEATHER_API_KEY= os.getenv('OPEN_WEATHER_API_KEY')
URL_BASE = "http://api.openweathermap.org/data/2.5/"

#go through cities
rain_sum = []
rain_prob_mean = []
humidity_mean = []
temp_feels_like_mean = []
wind_speed_mean = []
clouds_mean = []
 
for id, city in enumerate(cities):
    params = {
        'lat': lat_list[id],
        'lon': lon_list[id],
        'units': 'metric',
        'mode': 'json',
        'appid': OPEN_WEATHER_API_KEY, 
    }
    weather_data = requests.get(URL_BASE + 'forecast', params=params).json()
 
    if int(weather_data['cod']) == 200:
        dt_list = []
        temp_feels_like_list = []
        humidity_list = []
        wind_list = []
        pro_rain_list = []
        rain_list = []
        clouds_list = []
 
        for elem in range(weather_data['cnt']):
            dt_list.append(weather_data['list'][elem]['dt_txt'])
            pro_rain_list.append(weather_data['list'][elem]['pop'])
            if 'rain' in weather_data['list'][elem]:
                rain_list.append(weather_data['list'][elem]['rain']['3h'])
            else:
                rain_list.append(0.0)
            temp_feels_like_list.append(weather_data['list'][elem]['main']['feels_like'])
            humidity_list.append(weather_data['list'][elem]['main']['humidity'])
            wind_list.append(weather_data['list'][elem]['wind']['speed'])
            clouds_list.append(weather_data['list'][elem]['clouds']['all'])
 
        weather_df = pd.DataFrame(index=pd.to_datetime(dt_list))
        weather_df['temp_feels_like'] = list(np.float64(temp_feels_like_list))
        weather_df['humidity'] = list(np.float64(humidity_list))
        weather_df['wind_speed'] = list(np.float64(wind_list))
        weather_df['clouds'] = list(np.float64(clouds_list))
        weather_df['rain_probability'] = list(np.float64(pro_rain_list))
        weather_df['rain'] = list(np.float64(rain_list))
 
        day_weather_df = weather_df.between_time('09:00:00', '21:00:00', inclusive='both')
 
        rain_sum.append(day_weather_df['rain'].sum())
        rain_prob_mean.append(day_weather_df['rain_probability'].mean())
        humidity_mean.append(day_weather_df['humidity'].mean())
        temp_feels_like_mean.append(day_weather_df['temp_feels_like'].mean())
        wind_speed_mean.append(day_weather_df['wind_speed'].mean())
        clouds_mean.append(day_weather_df['clouds'].mean())
        print(f"{city}")
    else:
        print(f"{city} — API error: {weather_data.get('message', weather_data['cod'])}")
        rain_sum.append(None)
        rain_prob_mean.append(None)
        humidity_mean.append(None)
        temp_feels_like_mean.append(None)
        wind_speed_mean.append(None)
        clouds_mean.append(None)
 
df['rain'] = rain_sum
df['rain_probability'] = rain_prob_mean
df['humidity'] = humidity_mean
df['temp_feels_like'] = temp_feels_like_mean
df['wind_speed'] = wind_speed_mean
df['clouds'] = clouds_mean
 
print(df.head())
 

Le Mont-Saint-Michel
Saint Malo
Bayeux
Le Havre
Rouen
Paris
Amiens
Lille
Strasbourg
Chateau du Haut Koenigsbourg
Colmar
Eguisheim
Besancon
Dijon
Annecy
Grenoble
Lyon
Gorges du Verdon
Bormes les Mimosas
Cassis
Marseille
Aix en Provence
Avignon
Uzes
Nimes
Aigues Mortes
Saintes Maries de la mer
Collioure
Carcassonne
Ariege
Toulouse
Montauban
Biarritz
Bayonne
La Rochelle
                   city   latitude  longitude  rain  rain_probability  \
0  Le Mont-Saint-Michel  48.635523  -1.510257  2.24            0.1052   
1            Saint Malo  48.649518  -2.026041  3.51            0.1188   
2                Bayeux  49.276462  -0.702474  1.03            0.0480   
3              Le Havre  49.493898   0.107973  1.38            0.0400   
4                 Rouen  49.440459   1.093966  0.00            0.0000   

   humidity  temp_feels_like  wind_speed  clouds  
0     80.40          14.2936      4.7508   65.04  
1     81.44          13.0360      5.1260   65.76  
2     74.80          13.9528      4.43

In [ ]:
#Rank each column
df['rain_rank']= df['rain'].rank(method = 'min', ascending = True)
df['rain_probability_rank']= df['rain_probability'].rank(method = 'min', ascending = True)
df['humidity_rank']= df['humidity'].rank(method = 'min', ascending = True)
df['temp_feels_like_rank']= df['temp_feels_like'].rank(method = 'max', ascending = False)
df['wind_speed_rank']= df['wind_speed'].rank(method = 'min', ascending = True)
df['clouds_rank']= df['clouds'].rank(method = 'min', ascending = True)

#Selected criteria: min rain, min humidity, max temp
#Apply weights on selected criteria
weights = {'rain_rank': 0.5, 'humidity_rank': 0.15, 'temp_feels_like_rank': 0.35}
weighted_scores = df[['rain_rank', 'humidity_rank', 'temp_feels_like_rank']].mul(weights).sum(axis=1)
df['weighted_rank'] = weighted_scores.rank(method='min', ascending = True)

# Best location: 
print("Best location based on the criteria is:", df.loc[df['weighted_rank'].idxmin()].iloc[1])

print("5 Best locations based on the criteria is:")
display(df.sort_values(by=['weighted_rank'])[:5])

print("10 Best locations based on the criteria is:")
display(df.sort_values(by=['weighted_rank'])[:10])

df.to_json("weather.json", orient="records", force_ascii=False, indent=2)
print("Saved to weather.json")

Best location based on the criteria is: 43.8374249
5 Best locations based on the criteria is:


,city,latitude,longitude,rain,rain_probability,humidity,temp_feels_like,wind_speed,clouds,rain_rank,rain_probability_rank,humidity_rank,temp_feels_like_rank,wind_speed_rank,clouds_rank,weighted_rank
24,Nimes,43.837425,4.360069,0.0,0.0,45.88,19.2464,2.9736,14.12,1.0,1.0,3.0,2.0,15.0,2.0,1.0
22,Avignon,43.949249,4.805901,0.0,0.0,47.52,19.1932,3.3848,22.88,1.0,1.0,4.0,3.0,20.0,9.0,2.0
15,Grenoble,45.187560,5.735782,0.0,0.0,57.48,19.5672,1.6260,29.44,1.0,1.0,14.0,1.0,2.0,18.0,3.0
23,Uzes,44.012128,4.419672,0.0,0.0,48.12,18.7324,3.0300,16.00,1.0,1.0,5.0,5.0,17.0,6.0,4.0
27,Collioure,42.525050,3.083155,0.0,0.0,56.16,19.1444,2.9584,25.36,1.0,1.0,10.0,4.0,14.0,12.0,5.0


10 Best locations based on the criteria is:


,city,latitude,longitude,rain,rain_probability,humidity,temp_feels_like,wind_speed,clouds,rain_rank,rain_probability_rank,humidity_rank,temp_feels_like_rank,wind_speed_rank,clouds_rank,weighted_rank
24,Nimes,43.837425,4.360069,0.0,0.0,45.88,19.2464,2.9736,14.12,1.0,1.0,3.0,2.0,15.0,2.0,1.0
22,Avignon,43.949249,4.805901,0.0,0.0,47.52,19.1932,3.3848,22.88,1.0,1.0,4.0,3.0,20.0,9.0,2.0
15,Grenoble,45.187560,5.735782,0.0,0.0,57.48,19.5672,1.6260,29.44,1.0,1.0,14.0,1.0,2.0,18.0,3.0
23,Uzes,44.012128,4.419672,0.0,0.0,48.12,18.7324,3.0300,16.00,1.0,1.0,5.0,5.0,17.0,6.0,4.0
27,Collioure,42.525050,3.083155,0.0,0.0,56.16,19.1444,2.9584,25.36,1.0,1.0,10.0,4.0,14.0,12.0,5.0
21,Aix en Provence,43.529842,5.447474,0.0,0.0,39.44,18.5432,2.6972,19.04,1.0,1.0,1.0,8.0,11.0,7.0,6.0
5,Paris,48.858890,2.320041,0.0,0.0,43.04,17.9404,3.0100,43.80,1.0,1.0,2.0,11.0,16.0,22.0,7.0
16,Lyon,45.757814,4.832011,0.0,0.0,55.36,18.4544,2.7468,27.32,1.0,1.0,8.0,10.0,12.0,15.0,8.0
20,Marseille,43.296174,5.369953,0.0,0.0,57.16,17.0128,3.2336,13.52,1.0,1.0,12.0,13.0,19.0,1.0,9.0
14,Annecy,45.899235,6.128885,0.0,0.0,57.52,16.8084,1.4860,26.88,1.0,1.0,15.0,15.0,1.0,14.0,10.0


Saved to weather.json


# Scraping Booking.com

In [8]:
filename = "cities.json"
if filename in os.listdir():
        os.remove(filename)

with open(filename, 'w', encoding='utf-8') as f:
    json.dump(cities, f, ensure_ascii=False, indent=4)

In [15]:
!python web_scraping_project_kayak.py

Scraping 35 cities sequentially...

  [Le Mont-Saint-Michel] Found 5 hotels
  [Saint Malo] Found 5 hotels
  [Bayeux] Found 5 hotels
  [Le Havre] Found 5 hotels
  [Rouen] Found 5 hotels
  [Paris] Found 5 hotels
  [Amiens] ERROR 500 — skipping
  [Lille] Found 5 hotels
  [Strasbourg] Found 0 hotels
  [Chateau du Haut Koenigsbourg] ERROR 500 — skipping
  [Colmar] Found 0 hotels
  [Eguisheim] ERROR 500 — skipping
  [Besancon] Found 5 hotels
  [Dijon] Found 5 hotels
  [Annecy] Found 5 hotels
  [Grenoble] Found 5 hotels
  [Lyon] ERROR 500 — skipping
  [Gorges du Verdon] ERROR 500 — skipping
  [Bormes les Mimosas] Found 5 hotels
  [Cassis] Found 5 hotels
  [Marseille] Found 5 hotels
  [Aix en Provence] Found 5 hotels
  [Avignon] Found 5 hotels
  [Uzes] Found 5 hotels
  [Nimes] Found 0 hotels
  [Aigues Mortes] Found 5 hotels
  [Saintes Maries de la mer] ERROR 500 — skipping
  [Collioure] ERROR 500 — skipping
  [Carcassonne] Found 5 hotels
  [Ariege] ERROR 500 — skipping
  [Toulouse] Found 5 hot

In [ ]:
#Parse note and distance from hotels_df
def parse_note(text):
    if not text:
        return None
    # Extract first number like "8,2" or "8.2" after "note de"
    match = re.search(r'note de\s*([\d][,\.][\d])', str(text))
    if match:
        return float(match.group(1).replace(',', '.'))
    return None
 
def parse_distance(text):
    if not text:
        return None
    # Match "1,3 km" or "1.3 km" -> keep as km
    km_match = re.search(r'([\d]+[,\.]?[\d]*)\s*km', str(text))
    if km_match:
        return float(km_match.group(1).replace(',', '.'))
    # Match "200 m" -> convert to km
    m_match = re.search(r'([\d]+)\s*m', str(text))
    if m_match:
        return round(int(m_match.group(1)) / 1000, 2)
    return None

In [17]:
with open("hotels.json") as f:
    hotels = json.load(f)

hotels_df = pd.DataFrame(hotels).rename(columns={'ville': 'city'})
 
# Parse note and distance into numeric columns
hotels_df['note_score']   = hotels_df['note'].apply(parse_note)
hotels_df = hotels_df.drop(columns=['note'])
hotels_df['distance_km']  = hotels_df['distance'].apply(parse_distance)
hotels_df = hotels_df.drop(columns=['distance'])
#merge with df to get latitude, longitude and weighted_rank for maps
hotels_df = hotels_df.merge(
    df[['city', 'latitude', 'longitude', 'weighted_rank']],
    on='city', how='left'
)

hotels_df.head(20)

,city,nom,url,adresse,note_score,distance_km,latitude,longitude,weighted_rank
0,Le Mont-Saint-Michel,La Vieille Auberge,https://www.booking.com/hotel/fr/la-vieille-au...,Le Mont-Saint-Michel,7.5,0.03,48.635523,-1.510257,33.0
1,Le Mont-Saint-Michel,Le Duguesclin,https://www.booking.com/hotel/fr/le-duguesclin...,Le Mont-Saint-Michel,7.9,0.03,48.635523,-1.510257,33.0
2,Le Mont-Saint-Michel,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,Le Mont-Saint-Michel,8.3,2.40,48.635523,-1.510257,33.0
3,Le Mont-Saint-Michel,Hotel De La Digue,https://www.booking.com/hotel/fr/de-la-digue.f...,Le Mont-Saint-Michel,8.1,2.10,48.635523,-1.510257,33.0
4,Le Mont-Saint-Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html?...,Le Mont-Saint-Michel,8.2,2.40,48.635523,-1.510257,33.0
5,Saint Malo,T2 Confort Sillon St Malo,https://www.booking.com/hotel/fr/t2-duplex-cos...,"Sillon, Saint-Malo",7.8,2.10,48.649518,-2.026041,35.0
6,Saint Malo,Le Grand Sillon,https://www.booking.com/hotel/fr/le-grand-sill...,"Sillon, Saint-Malo",8.9,1.20,48.649518,-2.026041,35.0
7,Saint Malo,Escale Oceania Saint Malo,https://www.booking.com/hotel/fr/mascottesaint...,"Sillon, Saint-Malo",8.3,1.30,48.649518,-2.026041,35.0
8,Saint Malo,Le Pilo,https://www.booking.com/hotel/fr/le-pilori.fr....,"Intra muros, Saint-Malo",9.8,0.05,48.649518,-2.026041,35.0
9,Saint Malo,Les Bois Flottés,https://www.booking.com/hotel/fr/les-bois-flot...,"Paramé, Saint-Malo",9.2,3.30,48.649518,-2.026041,35.0


# Data lake - S3

In [18]:
S3_BUCKET ="pybnet-s3"
S3_RAW_FOLDER = "trip_Kayak/data/".rstrip("/") + "/"
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID") 
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = "eu-central-1" 

#Create client S3
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION,
)

# Check if folder exists, create it if not
existing = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=S3_RAW_FOLDER, MaxKeys=1)
if "Contents" in existing:
    print(f"Folder already exists: s3://{S3_BUCKET}/{S3_RAW_FOLDER}")
else:
    s3.put_object(Bucket=S3_BUCKET, Key=S3_RAW_FOLDER, Body=b"")
    print(f"Created folder: s3://{S3_BUCKET}/{S3_RAW_FOLDER}")
 
# Export df to CSV in memory and upload with timestamp
timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
s3_key = f"{S3_RAW_FOLDER}weather_{timestamp}.csv"

# Upload hotels.json to S3
csv_buffer = io.StringIO()
df.to_csv(csv_buffer, index=False, encoding="utf-8")

s3.put_object(
    Bucket=S3_BUCKET,
    Key=s3_key,
    Body=csv_buffer.getvalue().encode("utf-8"),
    ContentType="text/csv",
)
print(f"Uploaded weather CSV: {s3_key}")

# Upload hotels_df to S3
hotels_buffer = io.StringIO()
hotels_df.to_csv(hotels_buffer, index=False, encoding="utf-8")
s3.put_object(
    Bucket=S3_BUCKET,
    Key=f"{S3_RAW_FOLDER}hotels_{timestamp}.csv",
    Body=hotels_buffer.getvalue().encode("utf-8"),
    ContentType="text/csv",
)
print(f"Uploaded hotels CSV: {S3_RAW_FOLDER}hotels_{timestamp}.csv")

Folder already exists: s3://pybnet-s3/trip_Kayak/data/
Uploaded weather CSV: trip_Kayak/data/weather_20260407_072106.csv
Uploaded hotels CSV: trip_Kayak/data/hotels_20260407_072106.csv


# ETL

In [19]:
NEON_DB_URL = os.getenv("NEON_DB_URL")
WEATHER_TABLE_NAME = "trip_kayak_weather"
HOTELS_TABLE_NAME = "trip_kayak_hotels"

type_map = {
    "int64":   "INTEGER",
    "float64": "FLOAT",
    "bool":    "BOOLEAN",
    "object":  "TEXT",
}

def load_table(cur, table_name, dataframe):
    cur.execute(f"DROP TABLE IF EXISTS {table_name}")
    cols_sql = ", ".join(
        f'"{col}" {type_map.get(str(dataframe[col].dtype), "TEXT")}'
        for col in dataframe.columns
    )
    cur.execute(f"CREATE TABLE {table_name} ({cols_sql})")
    rows = [tuple(None if pd.isna(v) else v for v in row) for row in dataframe.itertuples(index=False)]
    placeholders = ", ".join(["%s"] * len(dataframe.columns))
    cur.executemany(f"INSERT INTO {table_name} VALUES ({placeholders})", rows)
    print(f"'{table_name}': {len(rows)} rows inserted")

conn = psycopg2.connect(NEON_DB_URL)
cur  = conn.cursor()

load_table(cur, WEATHER_TABLE_NAME, df)
load_table(cur, HOTELS_TABLE_NAME,  hotels_df)

conn.commit()
cur.close()
conn.close()
print("Done")

'trip_kayak_weather': 35 rows inserted
'trip_kayak_hotels': 115 rows inserted
Done


# Cartes
Les meilleurs hotels sont affichés pour le top des destinations.

In [ ]:
#Top 5 cities by weighted_rank, best hotel per city by note_score
top5_cities  = df.nsmallest(5, 'weighted_rank')
other_cities = df[~df['city'].isin(top5_cities['city'])]
 
top5_best_hotels = (
    hotels_df[hotels_df['city'].isin(top5_cities['city'])]
    .sort_values('note_score', ascending=False)
    .groupby('city')
    .first()
    .reset_index()
)
 
#Top 20 hotels overall
top20_hotels = hotels_df.sort_values('weighted_rank').head(20).reset_index(drop=True)
top20_hotels['rank'] = top20_hotels.index + 1
 
#Build side-by-side maps
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Top 5 destinations + meilleur hôtel', 'Top 20 hôtels'),
    specs=[[{"type": "geo"}, {"type": "geo"}]]
)
 
# Map 1 — other cities in gray
fig.add_trace(go.Scattergeo(
    lat=other_cities['latitude'],
    lon=other_cities['longitude'],
    mode='markers+text',
    marker=dict(size=8, color='#B4B2A9', line=dict(width=1, color='white')),
    text=other_cities['city'],
    textposition='top center',
    textfont=dict(size=9, color='#888780'),
    customdata=other_cities[['city', 'weighted_rank', 'temp_feels_like', 'rain', 'humidity']].values,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Rang: #%{customdata[1]:.0f}<br>"
        "Temp ressentie: %{customdata[2]:.1f}°C<br>"
        "Pluie: %{customdata[3]:.1f}mm<br>"
        "Humidité: %{customdata[4]:.0f}%"
        "<extra></extra>"
    ),
    name='Autres villes', showlegend=True,
), row=1, col=1)
 
# Map 1 — top 5 cities in teal
fig.add_trace(go.Scattergeo(
    lat=top5_cities['latitude'],
    lon=top5_cities['longitude'],
    mode='markers+text',
    marker=dict(size=16, color='#1D9E75', line=dict(width=2, color='white')),
    text=top5_cities['city'],
    textposition='top center',
    textfont=dict(size=11, color='#0F6E56'),
    customdata=top5_cities[['city', 'weighted_rank', 'temp_feels_like', 'rain', 'humidity']].values,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Rang: #%{customdata[1]:.0f}<br>"
        "Temp ressentie: %{customdata[2]:.1f}°C<br>"
        "Pluie: %{customdata[3]:.1f}mm<br>"
        "Humidité: %{customdata[4]:.0f}%"
        "<extra></extra>"
    ),
    name='Top 5', showlegend=True,
), row=1, col=1)
 
# Map 1 — best hotel per top 5 city in blue
fig.add_trace(go.Scattergeo(
    lat=top5_best_hotels['latitude'],
    lon=top5_best_hotels['longitude'],
    mode='markers',
    marker=dict(size=10, color='#378ADD', symbol='star', line=dict(width=1, color='white')),
    customdata=top5_best_hotels[['nom', 'city', 'note_score', 'adresse', 'distance_km']].values,
    hovertemplate=(
        "<b>⭐ %{customdata[0]}</b><br>"
        "%{customdata[1]}<br>"
        "Note: %{customdata[2]}<br>"
        "Adresse: %{customdata[3]}<br>"
        "Distance: %{customdata[4]} km"
        "<extra></extra>"
    ),
    name='Meilleur hôtel', showlegend=True,
), row=1, col=1)
 
# Map 2 — top 5 city labels
fig.add_trace(go.Scattergeo(
    lat=top5_cities['latitude'],
    lon=top5_cities['longitude'],
    mode='markers+text',
    marker=dict(size=14, color='#D85A30', symbol='square', line=dict(width=2, color='white')),
    text=top5_cities['city'],
    textposition='top center',
    textfont=dict(size=11, color='#993C1D'),
    hovertemplate="<b>%{text}</b><extra></extra>",
    name='Ville', showlegend=True,
), row=1, col=2)
 
# Map 2 — all hotels colored by weighted_rank
fig.add_trace(go.Scattergeo(
    lat=hotels_df['latitude'],
    lon=hotels_df['longitude'],
    mode='markers',
    marker=dict(
        size=10,
        color=hotels_df['weighted_rank'],
        colorscale='RdYlGn_r',
        reversescale=False,
        cmin=hotels_df['weighted_rank'].min(),
        cmax=hotels_df['weighted_rank'].max(),
        colorbar=dict(
            title='Classement pondéré',
            thickness=12,
            len=0.5,
            x=1.0,
            tickfont=dict(size=10),
        ),
        line=dict(width=1, color='white'),
    ),
    customdata=hotels_df[['nom', 'city', 'note_score', 'adresse', 'distance_km', 'weighted_rank']].values,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "%{customdata[1]}<br>"
        "Classement pondéré: #%{customdata[5]:.0f}<br>"
        "Note: %{customdata[2]}<br>"
        "Adresse: %{customdata[3]}<br>"
        "Distance: %{customdata[4]} km"
        "<extra></extra>"
    ),
    name='Hôtels', showlegend=True,
), row=1, col=2)
 
fig.update_layout(
    height=550,
    margin=dict(l=0, r=0, t=40, b=0),
    legend=dict(orientation='h', y=-0.05),
    geo=dict(
        scope='europe',
        center=dict(lat=46.5, lon=3.5),
        projection_scale=6,
        showland=True, landcolor='#EAF3DE',
        showocean=True, oceancolor='#E6F1FB',
        showcoastlines=True, coastlinecolor='#B4B2A9',
        showcountries=True, countrycolor='#D3D1C7',
        bgcolor='rgba(0,0,0,0)',
    ),
    geo2=dict(
        scope='europe',
        center=dict(lat=46.5, lon=3.5),
        projection_scale=6,
        showland=True, landcolor='#EAF3DE',
        showocean=True, oceancolor='#E6F1FB',
        showcoastlines=True, coastlinecolor='#B4B2A9',
        showcountries=True, countrycolor='#D3D1C7',
        bgcolor='rgba(0,0,0,0)',
    ),
)
 
fig.show()